In [3]:
# ================================================
# RAG 評估 + Gold(Y/N/F) 標註整合（只用 gold_labels.csv）
# ================================================
!pip -q install ragas litellm google-generativeai datasets evaluate pandas

import os, json, re, getpass
import pandas as pd
from pathlib import Path
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import llm_factory
from openai import OpenAI   # ✅ 新增：OpenAI client

# -------- 路徑（改這裡） --------
from google.colab import files
from pathlib import Path
import shutil

# 建 eval_data / eval_data/generated 資料夾
BASE = Path("./eval_data")
GEN_DIR = BASE / "generated"
BASE.mkdir(parents=True, exist_ok=True)
GEN_DIR.mkdir(parents=True, exist_ok=True)

# ① 上傳 generated_results.csv
print("請先上傳：generated_results.csv（RAG 批次結果）")
uploaded = files.upload()  # 在跳出的視窗選你本機的 generated_results.csv
gen_local_name = next(iter(uploaded.keys()))
gen_dst = GEN_DIR / "generated_results.csv"
shutil.move(gen_local_name, gen_dst)
print("已存到：", gen_dst)

# ② 上傳 gold_labels.csv
print("請再上傳：gold_labels.csv（Y/F/N gold 標）")
uploaded = files.upload()  # 選你本機的 gold_labels.csv
gold_local_name = next(iter(uploaded.keys()))
gold_dst = BASE / "gold_labels.csv"
shutil.move(gold_local_name, gold_dst)
print("已存到：", gold_dst)

# ③ 設定之後程式會用到的路徑變數
GEN_PATH  = gen_dst          # eval_data/generated/generated_results.csv
GOLD_PATH = gold_dst         # eval_data/gold_labels.csv

assert GEN_PATH.exists(), f"找不到 {GEN_PATH}"
assert GOLD_PATH.exists(), f"找不到 {GOLD_PATH}"

# -------- 讀 generated_results.csv --------
gen = pd.read_csv(GEN_PATH)

need_cols = {"qid","query","response","contexts"}
miss = need_cols - set(gen.columns)
if miss:
    raise ValueError(f"generated_results.csv 缺少欄位：{miss}")

def parse_contexts(x):
    """把 CSV 裡的 contexts 欄位轉回 list[str]（給 Ragas 用）"""
    if isinstance(x, list):
        return [str(t) for t in x]
    try:
        arr = json.loads(x)
        out = []
        if isinstance(arr, list):
            for it in arr:
                if isinstance(it, dict):
                    out.append(it.get("text", ""))
                else:
                    out.append(str(it))
            return out
    except Exception:
        pass
    return [str(x)]

gen["contexts_list"] = gen["contexts"].apply(parse_contexts)

# -------- 讀 gold_labels.csv（Y/N/F）並轉成文字 ground_truth --------
gold = pd.read_csv(GOLD_PATH)
if not {"query","suppose_answer"}.issubset(gold.columns):
    raise ValueError("gold_labels.csv 需至少包含 query, suppose_answer 欄位")

def ynf_to_text(y):
    """把 Y/N/F 映射成文字 ground_truth"""
    y = str(y).strip().upper()
    if y == "Y":
        return "是"
    if y == "N":
        return "否"
    # F：無法作答/資料外
    return "無相關資料"

gold_min = gold[["query","suppose_answer"]].copy()
gold_min["gold_answer_text"] = gold_min["suppose_answer"].map(ynf_to_text)

# 依 query 合併到 generated 結果
gen = gen.merge(gold_min[["query","gold_answer_text"]], on="query", how="left")

# ground_truth 完全用 gold_labels 的文字答案（沒標到就設成空字串）
gen["ground_truth"] = gen["gold_answer_text"].fillna("")

# -------- 輕量版：從模型回答 response 推回 Y/N/F（當成分類結果） --------
Y_PAT = r"(是|可以|會|有|屬於|為|用於)"
N_PAT = r"(不是|不可|不會|沒有|無法|非)"

def heuristic_label(a: str):
    s = (a or "").strip()
    if not s:
        return "F"
    # 先看明確否定字眼
    if re.search(N_PAT, s):
        return "N"
    # 再看肯定字眼
    if re.search(Y_PAT, s):
        return "Y"
    # 顯示拒答/無資料
    if "無相關資料" in s or "沒有資料" in s or "未見資料" in s:
        return "F"
    return "F"

gen["pred_label_from_resp"] = gen["response"].astype(str).apply(heuristic_label)

# 把 gold 的 suppose_answer（Y/N/F）也合併進來當真值
gold_label_map = gold_min.set_index("query")["suppose_answer"]
gen["gold_label"] = gen["query"].map(gold_label_map)

cls_acc = None
if gen["gold_label"].notna().any():
    cls_acc = (
        gen["pred_label_from_resp"].fillna("F").str.upper()
        == gen["gold_label"].fillna("F").str.upper()
    ).mean()

# -------- 準備 Ragas Dataset --------
ragas_df = pd.DataFrame({
    "question": gen["query"].astype(str),
    "contexts": gen["contexts_list"],
    "answer":   gen["response"].fillna("").astype(str),
    "ground_truth": gen["ground_truth"].astype(str),
})
ragas_ds = Dataset.from_pandas(ragas_df)

# -------- 評分用 LLM（使用 OpenAI + llm_factory，需要 client） --------
from openai import OpenAI
from ragas.llms import llm_factory

def pick_judge_llm():
    api_key = os.environ.get("OPENAI_API_KEY", "").strip()
    if not api_key:
        try:
            api_key = getpass.getpass("Enter OPENAI_API_KEY（用來做 Ragas 評分）: ").strip()
        except Exception:
            api_key = ""
        if api_key:
            os.environ["OPENAI_API_KEY"] = api_key

    api_key = os.environ.get("OPENAI_API_KEY", "").strip()
    if not api_key:
        raise ValueError("尚未設定 OPENAI_API_KEY，無法建立評分用 LLM。")

    # 建 OpenAI client，交給 ragas.llm_factory
    client = OpenAI(api_key=api_key)
    llm = llm_factory(model="gpt-4o", client=client)
    return llm

judge_llm = pick_judge_llm()

# ground_truth 有內容時即可評 context_recall
metrics = [faithfulness, answer_relevancy, context_precision]
if (ragas_df["ground_truth"].str.len() > 0).any():
    metrics.append(context_recall)

print("資料筆數：", len(ragas_ds))
print("使用指標：", [m.name for m in metrics])

scores = evaluate(
    dataset=ragas_ds,
    metrics=metrics,
    llm=judge_llm,
).to_pandas()

# -------- 輸出結果 --------
OUT = Path("./eval_out")
OUT.mkdir(exist_ok=True)

scores.to_csv(OUT / "ragas_scores.csv", index=False, encoding="utf-8-sig")

per_q = pd.concat(
    [
        gen[["qid","query","gold_label","pred_label_from_resp"]].reset_index(drop=True),
        scores.reset_index(drop=True)
    ],
    axis=1,
)
per_q.to_csv(OUT / "ragas_scores_per_question.csv", index=False, encoding="utf-8-sig")

print("完成。輸出：")
print(" -", OUT / "ragas_scores.csv")
print(" -", OUT / "ragas_scores_per_question.csv")
if cls_acc is not None:
    print(f" - Y/N/F 分類正確率（規則版）：{cls_acc:.3f}")

請先上傳：generated_results.csv（RAG 批次結果）


Saving generated_results.csv to generated_results.csv
已存到： eval_data/generated/generated_results.csv
請再上傳：gold_labels.csv（Y/F/N gold 標）


Saving gold_labels.csv to gold_labels.csv
已存到： eval_data/gold_labels.csv
資料筆數： 141
使用指標： ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']


Evaluating:   0%|          | 0/564 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[0]: AttributeError('InstructorLLM' object has no attribute 'agenerate_prompt')
ERROR:ragas.executor:Exception raised in Job[1]: AttributeError('InstructorLLM' object has no attribute 'agenerate_prompt')
ERROR:ragas.executor:Exception raised in Job[2]: AttributeError('InstructorLLM' object has no attribute 'agenerate_prompt')
ERROR:ragas.executor:Exception raised in Job[3]: AttributeError('InstructorLLM' object has no attribute 'agenerate_prompt')
ERROR:ragas.executor:Exception raised in Job[4]: AttributeError('InstructorLLM' object has no attribute 'agenerate_prompt')
ERROR:ragas.executor:Exception raised in Job[5]: AttributeError('InstructorLLM' object has no attribute 'agenerate_prompt')
ERROR:ragas.executor:Exception raised in Job[6]: AttributeError('InstructorLLM' object has no attribute 'agenerate_prompt')
ERROR:ragas.executor:Exception raised in Job[7]: AttributeError('InstructorLLM' object has no attribute 'agenerate_prompt')
ERROR:ra

✅ 完成。輸出：
 - eval_out/ragas_scores.csv
 - eval_out/ragas_scores_per_question.csv
 - Y/N/F 分類正確率（規則版）：0.624


In [4]:
from google.colab import files

# 分別下載兩個 CSV 檔
files.download("eval_out/ragas_scores.csv")
files.download("eval_out/ragas_scores_per_question.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>